# 02 EDA: KPI, Funnel, and Segment Analysis

This notebook performs exploratory analysis on processed tables to establish baseline KPIs, diagnose funnel drop-offs, compare segment behavior, and prepare hypotheses for A/B testing and uplift modeling.

**Expected processed inputs**
- `event_master.csv`
- `purchase_master.csv`
- `session_level_funnel.csv`
- `customer_level_features.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("📂 Loading the four core tables. Please wait...")

# 1. Load each file
# (These files are large, so loading may take a moment.)
event_master = pd.read_csv("../data/processed/event_master.csv")
purchase_master = pd.read_csv("../data/processed/purchase_master.csv")
session_df = pd.read_csv("../data/processed/session_level_funnel.csv")
customer_df = pd.read_csv("../data/processed/customer_level_features.csv")

# 2. Convert date columns back to datetime
# (This is necessary because dates become strings after CSV export.)
print("⏳ Converting date columns...")

# Convert event_master dates
date_cols_event = ['timestamp', 'signup_date', 'launch_date', 'start_date', 'end_date']
for col in date_cols_event:
    if col in event_master.columns:
        event_master[col] = pd.to_datetime(event_master[col])

# Convert purchase_master dates
if 'timestamp' in purchase_master.columns:
    purchase_master['timestamp'] = pd.to_datetime(purchase_master['timestamp'])

# Convert session_df dates
session_df['session_start_ts'] = pd.to_datetime(session_df['session_start_ts'])
session_df['session_end_ts'] = pd.to_datetime(session_df['session_end_ts'])

print("✅ Loading and conversion complete!")

# 3. Check table shapes to confirm successful loading
print(f"- event_master    : {event_master.shape}")
print(f"- purchase_master : {purchase_master.shape}")
print(f"- session_df      : {session_df.shape}")
print(f"- customer_df     : {customer_df.shape}")

In [ ]:
from IPython.display import display
# Adjust display settings for easier inspection (show all columns)
pd.set_option('display.max_columns', None)

data_frames = {
    "1. event_master (full event log)": event_master,
    "2. purchase_master (purchase receipt table)": purchase_master,
    "3. session_df (session summary table)": session_df,
    "4. customer_df (customer profile table)": customer_df
}

for name, df in data_frames.items():
    print(f"\n{'='*50}")
    print(f"📌 {name}")
    print(f"{'='*50}")
    print(f"- Shape: {df.shape}")
    print(f"- Columns: {df.columns.tolist()}")
    print("\n[Top 5 rows]")
    display(df.head(5))

In [ ]:
from IPython.display import display
# ================================================================
# PART 2 - STEP 1. Sanity Check
# Objective:
# 1) Negative-rate check for signup_to_event_days
# 2) Negative-rate check for product_age_days
# 3) Outlier-rate check for session duration
# 4) Quick check for whether outliers cluster in specific segments
# ================================================================


# ------------------------------------------------
# 0. Select input tables
# ------------------------------------------------
if "event_master" in globals():
    em = event_master.copy()
else:
    em = pd.read_csv("../data/processed/event_master.csv")
    print("📂 event_master.csv Loaded")

if "session_df" in globals():
    sdf = session_df.copy()
elif "session_level_funnel" in globals():
    sdf = session_level_funnel.copy()
else:
    sdf = pd.read_csv("../data/processed/session_level_funnel.csv")
    print("📂 session_level_funnel.csv Loaded")

# Fix date columns
for col in ["timestamp", "signup_date", "launch_date", "start_date", "end_date"]:
    if col in em.columns:
        em[col] = pd.to_datetime(em[col], errors="coerce")

for col in ["session_start_ts", "session_end_ts"]:
    if col in sdf.columns:
        sdf[col] = pd.to_datetime(sdf[col], errors="coerce")

# Fix numeric columns
for col in ["signup_to_event_days", "product_age_days", "session_duration_sec"]:
    if col in em.columns:
        em[col] = pd.to_numeric(em[col], errors="coerce")

for col in ["avg_session_duration_sec", "max_session_duration_sec", "wallclock_session_duration_sec"]:
    if col in sdf.columns:
        sdf[col] = pd.to_numeric(sdf[col], errors="coerce")

# ------------------------------------------------
# 1. Core sanity-check metrics
# ------------------------------------------------
n_events = len(em)
n_sessions = len(sdf)

neg_signup_mask = em["signup_to_event_days"] < 0
valid_product_age_mask = em["product_age_days"].notna()
neg_product_age_mask = valid_product_age_mask & (em["product_age_days"] < 0)

# Session duration outlier rules
# declared duration: session_duration_sec / max_session_duration_sec
# Simple rules:
#   - negative values
#   - 0
#   - above the 99.5th percentile (extreme values)
#   - severe mismatch between wallclock and declared duration
declared_col = "max_session_duration_sec" if "max_session_duration_sec" in sdf.columns else "avg_session_duration_sec"

declared_duration = sdf[declared_col]
wallclock_duration = sdf["wallclock_session_duration_sec"] if "wallclock_session_duration_sec" in sdf.columns else pd.Series(index=sdf.index, dtype=float)

p995_declared = declared_duration.quantile(0.995)
p995_wallclock = wallclock_duration.quantile(0.995) if wallclock_duration.notna().sum() > 0 else np.nan

neg_declared_mask = declared_duration < 0
zero_declared_mask = declared_duration == 0
long_declared_mask = declared_duration > p995_declared

if wallclock_duration.notna().sum() > 0:
    neg_wallclock_mask = wallclock_duration < 0
    long_wallclock_mask = wallclock_duration > p995_wallclock
    mismatch_mask = (declared_duration.notna()) & (wallclock_duration.notna()) & (
        np.abs(declared_duration - wallclock_duration) > 600  # difference greater than 10 minutes
    )
else:
    neg_wallclock_mask = pd.Series(False, index=sdf.index)
    long_wallclock_mask = pd.Series(False, index=sdf.index)
    mismatch_mask = pd.Series(False, index=sdf.index)

summary = pd.DataFrame({
    "metric": [
        "events rows",
        "sessions rows",
        "signup_to_event_days < 0",
        "product_age_days < 0 (valid only)",
        "declared session duration < 0",
        "declared session duration = 0",
        "declared session duration > p99.5",
        "wallclock session duration < 0",
        "wallclock session duration > p99.5",
        "declared vs wallclock mismatch (>600s)"
    ],
    "count": [
        n_events,
        n_sessions,
        int(neg_signup_mask.sum()),
        int(neg_product_age_mask.sum()),
        int(neg_declared_mask.sum()),
        int(zero_declared_mask.sum()),
        int(long_declared_mask.sum()),
        int(neg_wallclock_mask.sum()),
        int(long_wallclock_mask.sum()),
        int(mismatch_mask.sum())
    ],
    "ratio": [
        1.0,
        1.0,
        neg_signup_mask.mean(),
        neg_product_age_mask.mean() if valid_product_age_mask.sum() > 0 else np.nan,
        neg_declared_mask.mean(),
        zero_declared_mask.mean(),
        long_declared_mask.mean(),
        neg_wallclock_mask.mean() if len(sdf) > 0 else np.nan,
        long_wallclock_mask.mean() if len(sdf) > 0 else np.nan,
        mismatch_mask.mean() if len(sdf) > 0 else np.nan
    ]
})

print("\n==================== 1) Core sanity-check summary ====================")
display(summary)

# ------------------------------------------------
# 2. Where are negative signup_to_event_days values concentrated?
# ------------------------------------------------
print("\n==================== 2) signup_to_event_days < 0 segment check ====================")

signup_seg_cols = [c for c in ["country", "loyalty_tier", "acquisition_channel", "traffic_source", "device_type"] if c in em.columns]

signup_seg_tables = {}
for col in signup_seg_cols:
    tmp = (
        em.groupby(col, dropna=False)
          .agg(
              total_events=("event_id", "count"),
              neg_signup_events=("signup_to_event_days", lambda s: (s < 0).sum())
          )
          .reset_index()
    )
    tmp["neg_signup_rate"] = tmp["neg_signup_events"] / tmp["total_events"]
    tmp = tmp.sort_values(["neg_signup_rate", "neg_signup_events"], ascending=[False, False]).head(10)
    signup_seg_tables[col] = tmp

for col, tbl in signup_seg_tables.items():
    print(f"\n[Top 10 by {col}]")
    display(tbl)

# ------------------------------------------------
# 3. Where are negative product_age_days values concentrated?
# ------------------------------------------------
print("\n==================== 3) product_age_days < 0 segment check ====================")

prod_seg_cols = [c for c in ["category", "is_premium", "traffic_source", "page_category", "event_type"] if c in em.columns]

prod_seg_tables = {}
for col in prod_seg_cols:
    base = em.loc[em["product_age_days"].notna()].copy()
    tmp = (
        base.groupby(col, dropna=False)
            .agg(
                total_valid_events=("event_id", "count"),
                neg_product_age_events=("product_age_days", lambda s: (s < 0).sum())
            )
            .reset_index()
    )
    tmp["neg_product_age_rate"] = tmp["neg_product_age_events"] / tmp["total_valid_events"]
    tmp = tmp.sort_values(["neg_product_age_rate", "neg_product_age_events"], ascending=[False, False]).head(10)
    prod_seg_tables[col] = tmp

for col, tbl in prod_seg_tables.items():
    print(f"\n[Top 10 by {col}]")
    display(tbl)

# ------------------------------------------------
# 4. Summary of session-duration distributions
# ------------------------------------------------
print("\n==================== 4) Session duration distribution summary ====================")

duration_summary = pd.DataFrame({
    "metric": [
        "declared min", "declared p1", "declared p50", "declared p95", "declared p99", "declared p99.5", "declared max",
        "wallclock min", "wallclock p1", "wallclock p50", "wallclock p95", "wallclock p99", "wallclock p99.5", "wallclock max"
    ],
    "value": [
        declared_duration.min(),
        declared_duration.quantile(0.01),
        declared_duration.quantile(0.50),
        declared_duration.quantile(0.95),
        declared_duration.quantile(0.99),
        declared_duration.quantile(0.995),
        declared_duration.max(),
        wallclock_duration.min() if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.01) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.50) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.95) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.99) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.995) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.max() if wallclock_duration.notna().sum() > 0 else np.nan,
    ]
})
display(duration_summary)

# ------------------------------------------------
# 5. Where are session-duration outliers concentrated?
# ------------------------------------------------
print("\n==================== 5) Session-duration outlier segment check ====================")

sdf["duration_outlier_flag"] = (
    neg_declared_mask |
    zero_declared_mask |
    long_declared_mask |
    neg_wallclock_mask |
    long_wallclock_mask |
    mismatch_mask
).astype(int)

session_seg_cols = [c for c in ["first_device_type", "first_traffic_source", "first_page_category", "experiment_group", "funnel_stage"] if c in sdf.columns]

session_seg_tables = {}
for col in session_seg_cols:
    tmp = (
        sdf.groupby(col, dropna=False)
           .agg(
               total_sessions=("session_id", "count"),
               duration_outlier_sessions=("duration_outlier_flag", "sum")
           )
           .reset_index()
    )
    tmp["duration_outlier_rate"] = tmp["duration_outlier_sessions"] / tmp["total_sessions"]
    tmp = tmp.sort_values(["duration_outlier_rate", "duration_outlier_sessions"], ascending=[False, False]).head(10)
    session_seg_tables[col] = tmp

for col, tbl in session_seg_tables.items():
    print(f"\n[Top 10 by {col}]")
    display(tbl)

# ------------------------------------------------
# 6. Final summary for interpretation
# ------------------------------------------------
print("\n==================== 6) One-line interpretation summary ====================")
print(f"- signup_to_event_days < 0 Rate: {neg_signup_mask.mean():.2%}")
print(f"- product_age_days < 0 Rate: {neg_product_age_mask.mean() if valid_product_age_mask.sum() > 0 else np.nan:.2%}")
print(f"- declared duration outlier rate(negative/zero/>p99.5): {(neg_declared_mask | zero_declared_mask | long_declared_mask).mean():.2%}")
if wallclock_duration.notna().sum() > 0:
    print(f"- wallclock duration outlier rate(negative/>p99.5/mismatch): {(neg_wallclock_mask | long_wallclock_mask | mismatch_mask).mean():.2%}")

print("\n✅ STEP 1 complete. Next: STEP 2, build the overall KPI baseline table")

In [ ]:
# ================================================================
# PART 2 - STEP 2
# Overall KPI baseline + overall funnel
# ================================================================
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------
# 0. Prepare tables
# ------------------------------------------------
if "event_master" in globals():
    em = event_master.copy()
else:
    em = pd.read_csv("../data/processed/event_master.csv")

if "purchase_master" in globals():
    pm = purchase_master.copy()
else:
    pm = pd.read_csv("../data/processed/purchase_master.csv")

if "session_df" in globals():
    sdf = session_df.copy()
elif "session_level_funnel" in globals():
    sdf = session_level_funnel.copy()
else:
    sdf = pd.read_csv("../data/processed/session_level_funnel.csv")

if "customer_df" in globals():
    cdf = customer_df.copy()
elif "customer_level_features" in globals():
    cdf = customer_level_features.copy()
else:
    cdf = pd.read_csv("../data/processed/customer_level_features.csv")

# ------------------------------------------------
# 1. Basic cleanup
# ------------------------------------------------
for col in ["timestamp"]:
    if col in em.columns:
        em[col] = pd.to_datetime(em[col], errors="coerce")
    if col in pm.columns:
        pm[col] = pd.to_datetime(pm[col], errors="coerce")

pm["matched_tx_flag"] = pd.to_numeric(pm["matched_tx_flag"], errors="coerce").fillna(0)
pm["net_revenue"] = pd.to_numeric(pm["net_revenue"], errors="coerce").fillna(0)
pm["gross_revenue_filled"] = pd.to_numeric(pm["gross_revenue_filled"], errors="coerce").fillna(0)
pm["refund_flag_filled"] = pd.to_numeric(pm["refund_flag_filled"], errors="coerce").fillna(0)

# ------------------------------------------------
# 2. KPI baseline
# ------------------------------------------------
valid_orders = pm[pm["matched_tx_flag"] == 1].copy()

view_sessions = sdf["any_view"].sum()
click_sessions = sdf["any_click"].sum()
cart_sessions = sdf["any_add_to_cart"].sum()
purchase_sessions = sdf["any_purchase"].sum()

base_kpi = pd.DataFrame({
    "metric": [
        "Total customers",
        "Total sessions",
        "Total purchases (matched transactions)",
        "Session conversion rate",
        "Average sessions per customer",
        "Average orders per customer",
        "Gross revenue",
        "Net revenue",
        "Average order value (AOV)",
        "Refund rate",
        "Overall view→click rate",
        "Overall click→cart rate",
        "Overall cart→purchase rate"
    ],
    "value": [
        cdf["customer_id"].nunique(),
        len(sdf),
        len(valid_orders),
        sdf["is_converted"].mean(),
        cdf["total_sessions"].mean(),
        cdf["total_orders"].mean(),
        valid_orders["gross_revenue_filled"].sum(),
        valid_orders["net_revenue"].sum(),
        valid_orders["net_revenue"].mean(),
        valid_orders["refund_flag_filled"].mean(),
        click_sessions / view_sessions if view_sessions > 0 else np.nan,
        cart_sessions / click_sessions if click_sessions > 0 else np.nan,
        purchase_sessions / cart_sessions if cart_sessions > 0 else np.nan
    ]
})

print("\n==================== KPI Baseline ====================")
display(base_kpi)

# ------------------------------------------------
# 3. Overall funnel
# ------------------------------------------------
funnel_df = pd.DataFrame({
    "stage": ["view", "click", "add_to_cart", "purchase"],
    "sessions": [view_sessions, click_sessions, cart_sessions, purchase_sessions]
})
funnel_df["rate_from_prev"] = funnel_df["sessions"] / funnel_df["sessions"].shift(1)
funnel_df.loc[0, "rate_from_prev"] = 1.0

print("\n==================== Overall Funnel ====================")
display(funnel_df)

plt.figure(figsize=(8, 5))
plt.bar(funnel_df["stage"], funnel_df["sessions"])
plt.title("Overall Session Funnel")
plt.xlabel("Stage")
plt.ylabel("Number of Sessions")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(funnel_df["stage"], funnel_df["rate_from_prev"], marker="o")
plt.title("Stage-to-Stage Conversion Rate")
plt.xlabel("Stage")
plt.ylabel("Conversion Rate")
plt.ylim(0, 1.05)
plt.show()

In [ ]:
# ================================================================
# PART 2 - STEP 3
# Funnel comparison by device / traffic_source
# ================================================================
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

def funnel_by_segment(df, group_col):
    out = (
        df.groupby(group_col, dropna=False)
          .agg(
              total_sessions=("session_id", "count"),
              view_sessions=("any_view", "sum"),
              click_sessions=("any_click", "sum"),
              cart_sessions=("any_add_to_cart", "sum"),
              purchase_sessions=("any_purchase", "sum")
          )
          .reset_index()
    )
    out["session_conversion_rate"] = out["purchase_sessions"] / out["total_sessions"]
    out["view_to_click_rate"] = np.where(out["view_sessions"] > 0, out["click_sessions"] / out["view_sessions"], np.nan)
    out["click_to_cart_rate"] = np.where(out["click_sessions"] > 0, out["cart_sessions"] / out["click_sessions"], np.nan)
    out["cart_to_purchase_rate"] = np.where(out["cart_sessions"] > 0, out["purchase_sessions"] / out["cart_sessions"], np.nan)
    return out.sort_values("total_sessions", ascending=False)

device_funnel = funnel_by_segment(sdf, "first_device_type")
traffic_funnel = funnel_by_segment(sdf, "first_traffic_source")

print("\n==================== Funnel by Device ====================")
display(device_funnel)

print("\n==================== Funnel by Traffic Source ====================")
display(traffic_funnel)

# Session conversion by device
plt.figure(figsize=(8, 5))
plt.bar(device_funnel["first_device_type"].astype(str), device_funnel["session_conversion_rate"])
plt.title("Session Conversion Rate by Device")
plt.xlabel("Device")
plt.ylabel("Session Conversion Rate")
plt.show()

# Cart-to-purchase rate by device
plt.figure(figsize=(8, 5))
plt.bar(device_funnel["first_device_type"].astype(str), device_funnel["cart_to_purchase_rate"])
plt.title("Cart-to-Purchase Rate by Device")
plt.xlabel("Device")
plt.ylabel("Cart-to-Purchase Rate")
plt.show()

# Session conversion by traffic source
plt.figure(figsize=(8, 5))
plt.bar(traffic_funnel["first_traffic_source"].astype(str), traffic_funnel["session_conversion_rate"])
plt.title("Session Conversion Rate by Traffic Source")
plt.xlabel("Traffic Source")
plt.ylabel("Session Conversion Rate")
plt.show()

# Cart-to-purchase rate by traffic source
plt.figure(figsize=(8, 5))
plt.bar(traffic_funnel["first_traffic_source"].astype(str), traffic_funnel["cart_to_purchase_rate"])
plt.title("Cart-to-Purchase Rate by Traffic Source")
plt.xlabel("Traffic Source")
plt.ylabel("Cart-to-Purchase Rate")
plt.show()

In [ ]:
# ================================================================
# PART 2 - STEP 4
# Compare new vs existing customers + loyalty tiers + problem segments
# ================================================================
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

cdf2 = cdf.copy()

# ------------------------------------------------
# 1. Split into new vs existing customers
# signup_date is noisy, so use has_purchase_history instead
# ------------------------------------------------
cdf2["customer_status"] = np.where(
    cdf2["has_purchase_history"] == 1,
    "Existing_or_Buyer",
    "No_Purchase_History"
)

status_summary = (
    cdf2.groupby("customer_status", dropna=False)
        .agg(
            total_customers=("customer_id", "count"),
            total_sessions=("total_sessions", "sum"),
            converted_sessions=("converted_sessions", "sum"),
            total_orders=("total_orders", "sum"),
            net_revenue_total=("net_revenue_total", "sum"),
            avg_sessions_per_customer=("total_sessions", "mean"),
            avg_orders_per_customer=("total_orders", "mean"),
            avg_revenue_per_customer=("net_revenue_total", "mean")
        )
        .reset_index()
)

status_summary["session_conversion_rate"] = np.where(
    status_summary["total_sessions"] > 0,
    status_summary["converted_sessions"] / status_summary["total_sessions"],
    np.nan
)

status_summary["AOV"] = np.where(
    status_summary["total_orders"] > 0,
    status_summary["net_revenue_total"] / status_summary["total_orders"],
    np.nan
)

print("\n==================== New vs Existing Customer Comparison ====================")
display(status_summary)

plt.figure(figsize=(7, 5))
plt.bar(status_summary["customer_status"], status_summary["session_conversion_rate"])
plt.title("Session Conversion Rate by Customer Status")
plt.xlabel("Customer Status")
plt.ylabel("Session Conversion Rate")
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(status_summary["customer_status"], status_summary["AOV"])
plt.title("AOV by Customer Status")
plt.xlabel("Customer Status")
plt.ylabel("AOV")
plt.show()

# ------------------------------------------------
# 2. Compare loyalty tiers
# ------------------------------------------------
tier_order = ["Bronze", "Silver", "Gold", "Platinum"]

loyalty_summary = (
    cdf2.groupby("loyalty_tier", dropna=False)
        .agg(
            total_customers=("customer_id", "count"),
            total_sessions=("total_sessions", "sum"),
            converted_sessions=("converted_sessions", "sum"),
            total_orders=("total_orders", "sum"),
            net_revenue_total=("net_revenue_total", "sum"),
            refunded_orders=("refunded_orders", "sum"),
            premium_order_share=("premium_order_share", "mean"),
            avg_discount_sensitivity=("discount_sensitivity_score", "mean")
        )
        .reset_index()
)

loyalty_summary["session_conversion_rate"] = np.where(
    loyalty_summary["total_sessions"] > 0,
    loyalty_summary["converted_sessions"] / loyalty_summary["total_sessions"],
    np.nan
)

loyalty_summary["AOV"] = np.where(
    loyalty_summary["total_orders"] > 0,
    loyalty_summary["net_revenue_total"] / loyalty_summary["total_orders"],
    np.nan
)

loyalty_summary["refund_rate"] = np.where(
    loyalty_summary["total_orders"] > 0,
    loyalty_summary["refunded_orders"] / loyalty_summary["total_orders"],
    np.nan
)

loyalty_summary["loyalty_tier"] = pd.Categorical(
    loyalty_summary["loyalty_tier"], categories=tier_order, ordered=True
)
loyalty_summary = loyalty_summary.sort_values("loyalty_tier")

print("\n==================== Loyalty Tier Comparison ====================")
display(loyalty_summary)

plt.figure(figsize=(8, 5))
plt.bar(loyalty_summary["loyalty_tier"].astype(str), loyalty_summary["session_conversion_rate"])
plt.title("Session Conversion Rate by Loyalty Tier")
plt.xlabel("Loyalty Tier")
plt.ylabel("Session Conversion Rate")
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(loyalty_summary["loyalty_tier"].astype(str), loyalty_summary["AOV"])
plt.title("AOV by Loyalty Tier")
plt.xlabel("Loyalty Tier")
plt.ylabel("AOV")
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(loyalty_summary["loyalty_tier"].astype(str), loyalty_summary["refund_rate"])
plt.title("Refund Rate by Loyalty Tier")
plt.xlabel("Loyalty Tier")
plt.ylabel("Refund Rate")
plt.show()

# ------------------------------------------------
# 3. Customers who reached cart but did not purchase
# ------------------------------------------------
problem_customers = cdf2[
    (cdf2["total_add_to_cart"] > 0) &
    (cdf2["total_orders"] == 0)
].copy()

print("\n==================== Customers Who Added to Cart but Did Not Purchase ====================")
print(len(problem_customers))

problem_by_loyalty = (
    problem_customers.groupby("loyalty_tier", dropna=False)
        .agg(problem_customers=("customer_id", "count"))
        .reset_index()
        .sort_values("problem_customers", ascending=False)
)

problem_by_device = (
    problem_customers.groupby("preferred_device_type", dropna=False)
        .agg(problem_customers=("customer_id", "count"))
        .reset_index()
        .sort_values("problem_customers", ascending=False)
)

problem_by_traffic = (
    problem_customers.groupby("preferred_traffic_source", dropna=False)
        .agg(problem_customers=("customer_id", "count"))
        .reset_index()
        .sort_values("problem_customers", ascending=False)
)

problem_by_category = (
    problem_customers.groupby("favorite_category", dropna=False)
        .agg(problem_customers=("customer_id", "count"))
        .reset_index()
        .sort_values("problem_customers", ascending=False)
)

print("\n[Problem Segment - loyalty]")
display(problem_by_loyalty)

print("\n[Problem Segment - preferred device]")
display(problem_by_device)

print("\n[Problem Segment - preferred traffic]")
display(problem_by_traffic)

print("\n[Problem Segment - favorite category]")
display(problem_by_category.head(10))

In [ ]:
# ================================================================
# PART 2 - STEP 5
# Category performance diagnostics + first_purchase cohort
# ================================================================
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------
# 1. Behavioral funnel by category
# ------------------------------------------------
cat_event = (
    em[em["category"].notna()]
      .groupby("category", dropna=False)
      .agg(
          views=("is_view", "sum"),
          clicks=("is_click", "sum"),
          carts=("is_add_to_cart", "sum"),
          purchase_events=("is_purchase", "sum")
      )
      .reset_index()
)

# ------------------------------------------------
# 2. Revenue / refund analysis by category
# ------------------------------------------------
cat_purchase = (
    pm[(pm["matched_tx_flag"] == 1) & (pm["category"].notna())]
      .groupby("category", dropna=False)
      .agg(
          orders=("event_id", "count"),
          net_revenue=("net_revenue", "sum"),
          refunded_orders=("refund_flag_filled", "sum"),
          AOV=("net_revenue", "mean"),
          premium_share=("is_premium", lambda s: pd.Series(s).fillna(0).astype(float).mean())
      )
      .reset_index()
)

category_perf = cat_event.merge(cat_purchase, on="category", how="left")

category_perf["view_to_click_rate"] = np.where(
    category_perf["views"] > 0,
    category_perf["clicks"] / category_perf["views"],
    np.nan
)
category_perf["click_to_cart_rate"] = np.where(
    category_perf["clicks"] > 0,
    category_perf["carts"] / category_perf["clicks"],
    np.nan
)
category_perf["cart_to_purchase_rate"] = np.where(
    category_perf["carts"] > 0,
    category_perf["purchase_events"] / category_perf["carts"],
    np.nan
)
category_perf["refund_rate"] = np.where(
    category_perf["orders"] > 0,
    category_perf["refunded_orders"] / category_perf["orders"],
    np.nan
)

print("\n==================== Category Performance Diagnostics ====================")
display(category_perf.sort_values("orders", ascending=False))

plt.figure(figsize=(9, 5))
tmp = category_perf.sort_values("cart_to_purchase_rate", ascending=False)
plt.bar(tmp["category"], tmp["cart_to_purchase_rate"])
plt.title("Cart-to-Purchase Rate by Category")
plt.xlabel("Category")
plt.ylabel("Cart-to-Purchase Rate")
plt.show()

plt.figure(figsize=(9, 5))
tmp = category_perf.sort_values("refund_rate", ascending=False)
plt.bar(tmp["category"], tmp["refund_rate"])
plt.title("Refund Rate by Category")
plt.xlabel("Category")
plt.ylabel("Refund Rate")
plt.show()

plt.figure(figsize=(9, 5))
tmp = category_perf.sort_values("AOV", ascending=False)
plt.bar(tmp["category"], tmp["AOV"])
plt.title("AOV by Category")
plt.xlabel("Category")
plt.ylabel("AOV")
plt.show()

# ------------------------------------------------
# 3. repeat purchase cohort
# signup cohort is noisy, so use first_purchase cohort instead
# ------------------------------------------------
valid_orders = pm[pm["matched_tx_flag"] == 1].copy()
valid_orders["order_month"] = valid_orders["timestamp"].dt.to_period("M").astype(str)

first_purchase = (
    valid_orders.groupby("customer_id", as_index=False)
                .agg(first_purchase_ts=("timestamp", "min"))
)
first_purchase["first_purchase_month"] = first_purchase["first_purchase_ts"].dt.to_period("M").astype(str)

cohort_df = cdf[["customer_id", "total_orders", "net_revenue_total"]].merge(
    first_purchase[["customer_id", "first_purchase_month"]],
    on="customer_id",
    how="inner"
)

cohort_summary = (
    cohort_df.groupby("first_purchase_month", as_index=False)
             .agg(
                 total_customers=("customer_id", "count"),
                 repeat_buyers=("total_orders", lambda s: (s >= 2).sum()),
                 avg_orders_per_customer=("total_orders", "mean"),
                 avg_net_revenue_per_customer=("net_revenue_total", "mean")
             )
)

cohort_summary["repeat_purchase_rate"] = (
    cohort_summary["repeat_buyers"] / cohort_summary["total_customers"]
)

print("\n==================== first_purchase cohort ====================")
display(cohort_summary)

plt.figure(figsize=(10, 5))
plt.plot(cohort_summary["first_purchase_month"], cohort_summary["repeat_purchase_rate"], marker="o")
plt.title("Repeat Purchase Rate by First Purchase Cohort")
plt.xlabel("First Purchase Cohort Month")
plt.ylabel("Repeat Purchase Rate")
plt.xticks(rotation=45)
plt.show()